# ToolRL Reproduction Notebook

Provisions a Chameleon Cloud GPU instance, sets up Docker, and runs one training case end-to-end.

**Target:** GRPO [RL algorithm that doesn't need a separate critic model] Cold Start [trained straight from the base model, no supervised fine-tuning first], Qwen2.5-1.5B

**Paper result (API-Bank [the paper's own 597-question tool-call benchmark]):** 63.15% | **Our result:** 58.96%

**Wall time:** ~3.5 hours on 4x H100

**Before running:**
- Chameleon Cloud account with an active allocation at kvm.tacc.chameleoncloud.org
- SSH keypair registered in the KVM@TACC dashboard
- `pip install python-chi`

## 0. Configuration

In [ ]:
SITE         = "KVM@TACC"
PROJECT_NAME = "CHI-XXXXXX"       # your allocation number, e.g. CHI-231138
KEYPAIR_NAME = "my-chameleon-key" # as it appears in the dashboard at kvm.tacc.chameleoncloud.org
SSH_KEY_PATH = "~/.ssh/id_rsa"    # path to your private key inside JupyterHub
LEASE_HOURS  = 8
REPO_URL     = "https://github.com/Mario928/toolrl-verl-reproduction"

# --- GPU flavor: pick one based on what's available in your allocation ---
# FLAVOR = "gpu.h100.4x"   # 4x H100 NVL 94GB -- faster (~3.5h), set N_GPUS=4 below
# FLAVOR = "g1.h100.pci.1" # 1x H100 PCIe     -- slower (~7h),   set N_GPUS=1 below
FLAVOR = "gpu.h100.4x"    # change to g1.h100.pci.1 if 4x is unavailable

# --- GPU count: must match FLAVOR above ---
# N_GPUS = 1   # for g1.h100.pci.1
# N_GPUS = 4   # for gpu.h100.4x
N_GPUS = 4


## 1. Provision the Instance

**Using an existing active KVM@TACC lease:**
Cell 1a (lease creation) is commented out. Just run cell 1b — it launches the instance directly using the `FLAVOR` set in the config cell above.

If you don't have an active lease yet, uncomment cell 1a and run it first.

In [ ]:
# --- 1a: Create lease (SKIP if you already have an active KVM@TACC lease) ---
# Uncomment and run this cell only if you need a new lease.

# import chi, chi.lease, chi.server, chi.jupyterhub, datetime
#
# chi.use_site(SITE)
# chi.set("project_name", PROJECT_NAME)
# if chi.jupyterhub.is_jupyterhub_env():
#     chi.set("auth_type", "v3oidcaccesstoken")
#
# lease_obj = chi.lease.Lease(
#     "toolrl-reproduce",
#     duration=datetime.timedelta(hours=LEASE_HOURS),
# )
# lease_obj.add_flavor_reservation(name=FLAVOR, amount=1)
# lease_obj.submit(idempotent=True)
# print("Lease active:", lease_obj.id)

In [ ]:
# --- 1b: Launch instance ---
# Uses FLAVOR from the config cell above.
# If you ran cell 1a, you can also use: lease_obj.get_reserved_flavors()[0].name
import chi, chi.server, chi.jupyterhub

chi.use_site(SITE)
chi.set("project_name", PROJECT_NAME)
if chi.jupyterhub.is_jupyterhub_env():
    chi.set("auth_type", "v3oidcaccesstoken")

server = chi.server.create_server(
    "toolrl-node",
    flavor_name=FLAVOR,
    image_name="CC-Ubuntu22.04",
    key_name=KEYPAIR_NAME,
)
chi.server.wait_for_active(server.id)
floating_ip = chi.server.associate_floating_ip(server.id)
print("Floating IP:", floating_ip)

## 2. Connect via SSH

In [ ]:
import chi.ssh, os

node = chi.ssh.Remote(floating_ip, username="cc", key_filename=os.path.expanduser(SSH_KEY_PATH))
stdout, _ = node.execute("uname -a")
print(stdout)

## 3. Install Docker and nvidia-container-toolkit

In [ ]:
node.execute("sudo apt-get update -q")
node.execute("sudo apt-get install -y -q ca-certificates curl gnupg lsb-release")
node.execute("curl -fsSL https://download.docker.com/linux/ubuntu/gpg | sudo gpg --dearmor -o /usr/share/keyrings/docker-archive-keyring.gpg")
node.execute('echo "deb [arch=$(dpkg --print-architecture) signed-by=/usr/share/keyrings/docker-archive-keyring.gpg] https://download.docker.com/linux/ubuntu $(lsb_release -cs) stable" | sudo tee /etc/apt/sources.list.d/docker.list > /dev/null')
node.execute("sudo apt-get update -q")
node.execute("sudo apt-get install -y -q docker-ce docker-ce-cli containerd.io docker-compose-plugin")
node.execute("sudo usermod -aG docker cc")
print("Docker installed.")

In [ ]:
node.execute("curl -fsSL https://nvidia.github.io/libnvidia-container/gpgkey | sudo gpg --dearmor -o /usr/share/keyrings/nvidia-container-toolkit-keyring.gpg")
node.execute("curl -s -L https://nvidia.github.io/libnvidia-container/stable/deb/nvidia-container-toolkit.list | sed 's#deb https://#deb [signed-by=/usr/share/keyrings/nvidia-container-toolkit-keyring.gpg] https://#g' | sudo tee /etc/apt/sources.list.d/nvidia-container-toolkit.list")
node.execute("sudo apt-get update -q")
node.execute("sudo apt-get install -y -q nvidia-container-toolkit")
node.execute("sudo nvidia-ctk runtime configure --runtime=docker")
node.execute("sudo systemctl restart docker")
print("nvidia-container-toolkit installed.")

In [ ]:
stdout, _ = node.execute("sudo docker run --rm --gpus all nvidia/cuda:12.1.0-base-ubuntu22.04 nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
print(stdout)

## 4. Clone Repo and Create Volumes

In [ ]:
node.execute(f"git clone {REPO_URL} /home/cc/toolrl")
for vol in ["toolrl_models", "toolrl_hf_cache", "toolrl_mlflow_data", "toolrl_datasets"]:
    node.execute(f"sudo docker volume create {vol}")
print("Done.")

## 5. Build the Docker Image

Takes ~15-20 minutes the first time.

In [ ]:
stdout, _ = node.execute("cd /home/cc/toolrl && sudo docker build -t toolrl-verl:latest . 2>&1 | tail -20")
print(stdout)

## 6. Start Containers

In [ ]:
node.execute("cd /home/cc/toolrl && sudo docker compose up -d")
stdout, _ = node.execute("sudo docker ps --format 'table {{.Names}}\t{{.Status}}'")
print(stdout)

## 7. Prepare the Dataset

Converts the raw JSON files already in the repo into the parquet format the trainer expects. Must run before training or the trainer crashes at step 1 with `KeyError: 'reward_model'`.

In [ ]:
node.execute("sudo docker exec verl bash -c 'cd /workspace && python dataset/rlla_4k_raw/rlla.py'")
print("Dataset prepared.")

## 8. Download the Base Model

In [ ]:
node.execute(
    "sudo docker exec verl python3 -c "
    "'from huggingface_hub import snapshot_download; "
    "snapshot_download(\"Qwen/Qwen2.5-1.5B-Instruct\")'"
)
print("Model downloaded.")

## 9. Run Training

GRPO [RL algorithm, no critic model] cold start [no supervised fine-tuning beforehand], 15 epochs, 4 GPUs. Final checkpoint at `global_step_90`.

To run a different variant, change `BASE_MODEL`, `EXPERIMENT_NAME`, and the reward flags.

In [ ]:
node.execute(f"""
sudo docker exec -d verl bash -c '
    export CUDA_VISIBLE_DEVICES={ ','.join(str(i) for i in range(N_GPUS)) }
    export N_GPUS={N_GPUS}
    export ROLLOUT_TP_SIZE=1
    export VLLM_ATTENTION_BACKEND=XFORMERS
    export WITHLENGTH=0 REFINEDREWARD=0 COARSEREWARD=0 STRICTMATCH=0
    export CORRECTMAX1=0 MAX1STEP30MAX3=0 SCHEDULEREWARD=0 SCHEDULELENGTH=0
    export DATA_DIR="./dataset/rlla_4k"
    export BASE_MODEL="Qwen/Qwen2.5-1.5B-Instruct"
    export EXPERIMENT_NAME="/app/models/toolrl-grpo-cold-qwen-1.5b"
    cd /workspace && bash ./examples/grpo_trainer/run_grpo.sh > /tmp/train.log 2>&1
'
""")
print(f"Training started in background ({N_GPUS} GPU(s)). Run the next cell to check progress.")

In [ ]:
import time

# Poll training log every 60s and print latest lines
# Run this cell to check progress; re-run as needed
stdout, _ = node.execute("sudo docker exec verl tail -20 /tmp/train.log 2>/dev/null || echo 'Log not yet available'")
print(stdout)

## 10. Evaluate on API-Bank

Run after training finishes (~3.5 hours). Generation takes ~5 minutes for 597 questions.

In [ ]:
CHECKPOINT = "/app/models/toolrl-grpo-cold-qwen-1.5b/actor/global_step_90"

node.execute(f"""
sudo docker exec verl bash -c '
    cd /workspace/benchmarks/API-Bank &&
    WORLD_SIZE=4 python3 generate_batch.py --model_paths {CHECKPOINT} > /tmp/apibank_gen.log 2>&1
'
""")
print("Generation done.")

In [ ]:
stdout, _ = node.execute(f"""
sudo docker exec verl bash -c '
    cd /workspace/benchmarks/API-Bank &&
    python3 evaluate.py --model_paths {CHECKPOINT}
'
""")
print(stdout)

## 11. Cleanup

Run in order. Uncomment and run one cell at a time.

In [ ]:
# Stop containers
# node.execute("cd /home/cc/toolrl && sudo docker compose down")

In [ ]:
# Delete the server
# chi.server.delete_server(server.id)

In [ ]:
# Delete the lease
# lease_obj.delete()
# print("Lease deleted.")